# sakuhart

Turn a video or a photo into mosaic art, in the browser. Nothing to install on your machine.

Run the cells in order. The whole thing takes a few minutes.

## 1. Install

In [ ]:
!pip install -q sakuhart

import subprocess
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="To exit")  # IPython's reply to SystemExit


def run(*cmd):
    """Run a command and stop the notebook if it fails.

    A bare `!command` keeps going after an error, and the next cell then fails
    on a file that was never written."""
    proc = subprocess.Popen(
        [str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise SystemExit(f"{cmd[0]} failed -- see the message above.")


run("sakuhart", "--version")

## 2. Pick the photos your mosaic is made of

Each pack is a few thousand photographs, all CC BY 2.0. Pick the one whose colours suit your
picture: dogs are warm and brown, cats are dark and grey, flowers are bright.

In [ ]:
import urllib.request
import zipfile

pack = "dog"  # @param ["dog", "cat", "flower"]

url = f"https://github.com/hinanohart/sakuhart/releases/download/v0.1.0/{pack}-tiles.zip"
print(f"downloading {pack}-tiles.zip ...")
urllib.request.urlretrieve(url, "tiles.zip")
with zipfile.ZipFile("tiles.zip") as z:
    z.extractall()
print(f"{pack}: {len(list(Path(pack).glob('*.jpg')))} photos")

## 3. Upload your video or photo

A photo takes seconds. For video, a few seconds of footage is plenty to start with -- a
one-minute clip takes about ten minutes here.

In [ ]:
from google.colab import files

VIDEO = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}

uploaded = files.upload()
if not uploaded:
    raise SystemExit("nothing was uploaded -- run this cell again and pick a file.")
clip = next(iter(uploaded))
is_video = Path(clip).suffix.lower() in VIDEO
out = "mosaic.mp4" if is_video else "mosaic.png"
print(f"using {clip} -> {out}")

## 4. Build it

`--cells` is how many tiles the picture is cut into. 700 is the default; raise it for a finer
mosaic and a longer run.

In [ ]:
cells = 700  # @param {type:"integer"}

run("sakuhart", clip, "-t", pack, "--cells", cells, "-o", out)

## 5. See it, then download it

In [ ]:
from base64 import b64encode

from IPython.display import HTML, Image, display

mb = Path(out).stat().st_size / 1e6
if not is_video:
    display(Image(out, width=600))
elif mb < 25:
    src = "data:video/mp4;base64," + b64encode(open(out, "rb").read()).decode()
    display(HTML(f'<video width=600 controls loop><source src="{src}"></video>'))
else:
    print(f"{out} is {mb:.0f} MB -- too big to play inline; download it below.")

In [ ]:
files.download(out)

---

The pack you used is CC BY 2.0: if you publish the mosaic, ship the `ATTRIBUTION.md` inside the
zip alongside it. sakuhart itself is MIT.